In [ ]:
# Generated from dataops/02-audit.yaml -- do not edit by hand.
#
# One row per entity per layer per run: how many rows, how much value, and what
# the layer rejected. Written to the warehouse so Power BI reports on it beside
# the numbers it audits.

from datetime import datetime, timezone

from pyspark.sql import functions as F

PLAN = [{'entity': 'order_items',
  'layer': 'bronze',
  'table': 'bronze_commerce_order_items',
  'measure': 'subtotal',
  'filter': None,
  'quarantine': None,
  'kind': 'lakehouse',
  'item': 'lh_bronze',
  'grain_changes': False},
 {'entity': 'order_items',
  'layer': 'silver',
  'table': 'stg_order_items',
  'measure': 'subtotal',
  'filter': None,
  'quarantine': 'stg_order_items_quarantine',
  'kind': 'lakehouse',
  'item': 'lh_silver',
  'grain_changes': False},
 {'entity': 'order_items',
  'layer': 'gold',
  'table': 'dbo.fct_sales',
  'measure': 'line_revenue',
  'filter': None,
  'quarantine': None,
  'kind': 'warehouse',
  'item': 'wh_gold',
  'grain_changes': False},
 {'entity': 'orders',
  'layer': 'bronze',
  'table': 'bronze_commerce_orders',
  'measure': 'total_price',
  'filter': None,
  'quarantine': None,
  'kind': 'lakehouse',
  'item': 'lh_bronze',
  'grain_changes': False},
 {'entity': 'orders',
  'layer': 'silver',
  'table': 'stg_orders',
  'measure': 'order_total',
  'filter': None,
  'quarantine': 'stg_orders_quarantine',
  'kind': 'lakehouse',
  'item': 'lh_silver',
  'grain_changes': False},
 {'entity': 'customers',
  'layer': 'bronze',
  'table': 'bronze_commerce_customers',
  'measure': None,
  'filter': None,
  'quarantine': None,
  'kind': 'lakehouse',
  'item': 'lh_bronze',
  'grain_changes': True},
 {'entity': 'customers',
  'layer': 'silver',
  'table': 'stg_customers',
  'measure': None,
  'filter': None,
  'quarantine': 'stg_customers_quarantine',
  'kind': 'lakehouse',
  'item': 'lh_silver',
  'grain_changes': True},
 {'entity': 'customers',
  'layer': 'gold',
  'table': 'dbo.dim_customer',
  'measure': None,
  'filter': 'is_current = true',
  'quarantine': None,
  'kind': 'warehouse',
  'item': 'wh_gold',
  'grain_changes': True},
 {'entity': 'products',
  'layer': 'bronze',
  'table': 'bronze_commerce_products',
  'measure': 'price',
  'filter': None,
  'quarantine': None,
  'kind': 'lakehouse',
  'item': 'lh_bronze',
  'grain_changes': True},
 {'entity': 'products',
  'layer': 'silver',
  'table': 'stg_products',
  'measure': 'list_price',
  'filter': None,
  'quarantine': 'stg_products_quarantine',
  'kind': 'lakehouse',
  'item': 'lh_silver',
  'grain_changes': True},
 {'entity': 'products',
  'layer': 'gold',
  'table': 'dbo.dim_product',
  'measure': 'list_price',
  'filter': 'is_current = true',
  'quarantine': None,
  'kind': 'warehouse',
  'item': 'wh_gold',
  'grain_changes': True}]

WAREHOUSE = 'wh_gold'
TARGET_SCHEMA = 'dbo'
TARGET_TABLE = 'audit_layer_flow'
WRITE_MODE = 'append'

run_id = f"audit_{datetime.now(timezone.utc):%Y%m%d_%H%M%S}"
print(f"audit run {run_id}")
print()


DEFAULT_LAKEHOUSE = 'lh_silver'


def read(kind, item, name):
    """Read a table from whichever item that layer actually lives in.

    The lakehouse name is NOT optional here. An unqualified read resolves
    against this notebook's default lakehouse, so every bronze table would be
    looked for in silver and come back as an error -- the audit would report
    bronze as unmeasurable while silver and gold looked fine, which reads as a
    missing layer rather than a wrong lookup.
    """
    bare = name.rsplit(".", 1)[-1]
    if kind == "warehouse":
        # The connector reads the warehouse; spark.read.table cannot.
        import com.microsoft.spark.fabric                    # noqa: F401
        schema_name = name.rsplit(".", 2)[-2] if "." in name else "dbo"
        return spark.read.synapsesql(f"{item}.{schema_name}.{bare}")   # noqa: F821
    if item and item != DEFAULT_LAKEHOUSE:
        return spark.read.table(f"{item}.{bare}")         # noqa: F821
    return spark.read.table(bare)                            # noqa: F821


def measure(entry):
    """rows, value and rejected for one entity at one layer."""
    df = read(entry["kind"], entry["item"], entry["table"])
    if entry["filter"]:
        df = df.filter(entry["filter"])

    rows = df.count()
    value = None
    if entry["measure"]:
        total = df.agg(F.sum(F.col(entry["measure"])).alias("v")).collect()[0]["v"]
        value = float(total) if total is not None else 0.0

    rejected = None
    if entry["quarantine"]:
        # Absent rather than zero when there is no quarantine table: a layer
        # that has never rejected anything and a layer that cannot reject are
        # different states, and reporting both as 0 hides the second.
        try:
            rejected = read(entry["kind"], entry["item"], entry["quarantine"]).count()
        except Exception:                                    # noqa: BLE001
            rejected = None

    return rows, value, rejected


records = []
for entry in PLAN:
    try:
        rows, value, rejected = measure(entry)
        records.append((run_id, entry["entity"], entry["layer"], entry["table"],
                        rows, value, rejected, entry["grain_changes"], None,
                        datetime.now(timezone.utc)))
        shown = "-" if value is None else f"{value:,.2f}"
        print(f"  {entry['entity']:<14} {entry['layer']:<7} "
              f"rows={rows:>9,}  value={shown:>18}  "
              f"rejected={'-' if rejected is None else format(rejected, ',')}")
    except Exception as exc:                                 # noqa: BLE001
        # Recorded, not raised. An audit that stops at the first unreadable
        # table tells you nothing about the layers that were fine.
        records.append((run_id, entry["entity"], entry["layer"], entry["table"],
                        None, None, None, entry["grain_changes"],
                        f"{type(exc).__name__}: {exc}"[:400],
                        datetime.now(timezone.utc)))
        print(f"  {entry['entity']:<14} {entry['layer']:<7} FAILED  {str(exc)[:90]}")

schema = ("run_id string, entity string, layer string, table_name string, "
          "row_count bigint, measure_value double, rejected_count bigint, "
          "grain_changes boolean, error string, measured_at timestamp")
audit_df = spark.createDataFrame(records, schema)            # noqa: F821

# The connector is overwrite-only, so appending means read, union, overwrite.
if WRITE_MODE == "append":
    try:
        import com.microsoft.spark.fabric                    # noqa: F401
        existing = spark.read.synapsesql(                    # noqa: F821
            f"{WAREHOUSE}.{TARGET_SCHEMA}.{TARGET_TABLE}")
        audit_df = existing.unionByName(audit_df, allowMissingColumns=True)
    except Exception:                                        # noqa: BLE001
        print("\nno existing audit table; creating it")

import com.microsoft.spark.fabric                            # noqa: F401
from com.microsoft.spark.fabric.Constants import Constants   # noqa: F401
(audit_df.write.mode("overwrite")
    .synapsesql(f"{WAREHOUSE}.{TARGET_SCHEMA}.{TARGET_TABLE}"))

print(f"\n{len(records)} row(s) written to "
      f"{TARGET_SCHEMA}.{TARGET_TABLE} as {run_id}")
